# 03 - Deteccao de Anomalia de Vibracao (Autoencoder)

Autoencoder de reconstrucao (MLP) sobre janelas de vibracao. O limiar e o
percentil 99 do erro de reconstrucao em operacao normal. Ver
`ml/pipelines/train_anomaly.py` e ADR 0005.

In [ ]:
import sys
sys.path.insert(0, '../pipelines')
from train_anomaly import WINDOW_SIZE, train, _windows

meta = train()
meta

In [ ]:
import joblib
import numpy as np
from features import MODELS_DIR, load_dataset

model = joblib.load(MODELS_DIR / meta['artifact'])
df = load_dataset()
windows = _windows(df['vibration_velocity_rms'].to_numpy(dtype=float), WINDOW_SIZE)
errors = np.mean((windows - model.predict(windows)) ** 2, axis=1)
print('limiar:', meta['reconstruction_threshold'])
print('janelas acima do limiar:', int((errors > meta['reconstruction_threshold']).sum()))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(errors, linewidth=0.6)
ax.axhline(meta['reconstruction_threshold'], color='red', linestyle='--', label='limiar')
ax.set_title('Erro de reconstrucao do autoencoder')
ax.legend()
plt.show()